# 2 · Functions
*Intro to Python for Scientists & Public Health Professionals*

Functions package logic into named, reusable, testable pieces. If you've written functions in another language, the ideas transfer — this notebook focuses on Python's conventions (defaults, `*args`/`**kwargs`, lambdas) and a couple of traps.

### By the end of this notebook you can
- Define functions with positional, keyword, and default arguments
- Use `return` deliberately, including early-exit guard clauses and multiple return values
- Reason about local vs global scope
- Write `*args` / `**kwargs` functions and avoid the mutable-default-argument trap
- Use lambdas where they belong (as `key=`, with `map`/`filter`/`sorted`)

### Agenda
1. Defining and calling functions
2. Arguments: positional, keyword, defaults
3. Return values
4. Variable scope
5. Type hints
6. `*args` and `**kwargs`
7. Lambdas & higher-order functions
8. A practical example with pandas

### How the exercises work
Each exercise has a prompt, an empty cell to try it yourself, and a collapsed **Solution** you can expand to check your work.

## 1. Defining and calling functions

`def` introduces the definition, followed by the name and a parenthesized parameter list. The indented block below is the body. A function does nothing until it is **called**.

It helps to keep two words straight:
- A **parameter** is the placeholder named in the definition.
- An **argument** is the actual value you pass when calling.

In [ ]:
def do_math(a):              # 'a' is the parameter
    x = a + 10
    return x

do_math(5)                   # 5 is the argument

Give every non-trivial function a **docstring** — a string literal as its first line. Tools and `help()` read it.

In [ ]:
def coverage_rate(administered, eligible):
    """Return the fraction of eligible people who received a dose."""
    return administered / eligible

help(coverage_rate)

> Reference: [Defining functions](https://docs.python.org/3/tutorial/controlflow.html#defining-functions).

## 2. Arguments: positional, keyword, defaults

Arguments can be passed **by position** or **by keyword**, and parameters can carry **default values** that make them optional.

In [ ]:
def coverage(administered, eligible, as_percent=True):
    rate = administered / eligible
    return f"{rate:.1%}" if as_percent else rate

print(coverage(947, 1085))                    # positional
print(coverage(947, 1085, as_percent=False))  # override the default by keyword

One rule to remember: a **positional argument cannot follow a keyword argument**.

In [ ]:
# This is a SyntaxError. Uncomment to see it, then re-comment so the notebook runs.
# coverage(947, as_percent=False, 1085)

## 3. Return values

`return` sends a value back to the caller and **immediately ends** the function. A function with no `return` returns `None`.

In [ ]:
def log_event(msg):
    print(f"[log] {msg}")     # no return statement

result = log_event("clinic opened")
print(result)                 # None

**Guard clauses** use an early `return` to handle edge cases first and keep the main logic flat.

In [ ]:
def risk_band(a1c):
    if a1c is None:
        return "no reading"        # exit early
    if a1c >= 9.0:
        return "high"
    elif a1c >= 7.0:
        return "elevated"
    return "controlled"

print(risk_band(9.4), "|", risk_band(None))

A function can return **multiple values** as a tuple, which the caller unpacks (same idea as the tuple unpacking from Lesson 1).

In [ ]:
def summarize(values):
    """Return (min, max, mean) of a sequence of numbers."""
    return min(values), max(values), sum(values) / len(values)

lo, hi, avg = summarize([7.1, 6.8, 9.2, 5.5])
print(lo, hi, round(avg, 2))

### Exercise 1 — A reusable filter *(5 min)*

Write a function `high_readings(values, threshold)` that returns a new list of only the values greater than or equal to `threshold`. Test it on a few A1C readings with a cutoff of 7.0.

In [ ]:
# Your work here


<details>
<summary>Solution</summary>

```python
def high_readings(values, threshold):
    "Return readings at or above the threshold."
    return [v for v in values if v >= threshold]

print(high_readings([7.1, 6.8, 9.2, 5.5, 8.0], 7.0))
```

**Why this works.** Making `threshold` a parameter (rather than hard-coding `7.0`) is what makes the function reusable for any cutoff. The comprehension builds and returns a brand-new list, so the caller's original list is never modified.

</details>

## 4. Variable scope

A variable assigned **inside** a function is **local** — it exists only during the call. A variable defined at the top level is **global**.

In [ ]:
def add_ten(a):
    result = a + 10      # local: gone once the function returns
    return result

print(add_ten(5))
# print(result)          # would raise NameError — 'result' is local to add_ten

To *rebind* a global from inside a function you must declare it with `global`. Use this sparingly — globals mutated from inside functions are a common source of hard-to-trace bugs.

In [ ]:
counter = 0

def bump():
    global counter       # without this, 'counter += 1' would create a local
    counter += 1

bump(); bump()
print(counter)           # 2

## 5. Type hints

Optional annotations that document the expected types. They are **not enforced at runtime** — Python ignores them — but they make intent clear and power editor autocomplete, linters, and type checkers.

In [ ]:
def coverage_rate(administered: int, eligible: int) -> float:
    """Return the fraction of eligible people who received a dose."""
    return administered / eligible

print(coverage_rate(947, 1085))

> Reference: [`typing` module](https://docs.python.org/3/library/typing.html).

## 6. `*args` and `**kwargs`

`*args` collects extra **positional** arguments into a tuple; `**kwargs` collects extra **keyword** arguments into a dict. Use them when the number of inputs varies.

In [ ]:
def total_doses(*daily_counts):
    """Sum any number of daily dose counts."""
    total = 0
    for c in daily_counts:
        total += c
    return total

print(total_doses(120, 95, 130))        # three days
print(total_doses(120, 95, 130, 88))    # ...or four

In [ ]:
def bio(first, last, **details):
    age  = details.get("age", "unknown")       # .get() avoids KeyError if absent
    eyes = details.get("eyecolor", "unknown")
    return f"{first} {last} is {age} years old and has {eyes} eyes"

print(bio("Mary", "Smith", age=32, eyecolor="brown"))
print(bio("Mary", "Smith"))                     # no extras -> 'unknown', no crash

> Reference: [Arbitrary argument lists](https://docs.python.org/3/tutorial/controlflow.html#arbitrary-argument-lists).

### Gotcha — never use a mutable default argument

A default value is created **once**, when the function is defined, and then **shared across all calls**. A mutable default (like a list) therefore accumulates between calls — almost never what you want.

In [ ]:
def add_reading(value, readings=[]):    # BUG: the default list is shared
    readings.append(value)
    return readings

print(add_reading(7.1))   # [7.1]
print(add_reading(6.8))   # [7.1, 6.8]  <- the same list came back!

In [ ]:
def add_reading(value, readings=None):  # FIX: use None as a sentinel
    if readings is None:
        readings = []                   # a fresh list on every call
    readings.append(value)
    return readings

print(add_reading(7.1))   # [7.1]
print(add_reading(6.8))   # [6.8]

> Background: [Why are default values shared between objects?](https://docs.python.org/3/faq/programming.html#why-are-default-values-shared-between-objects)

## 7. Lambdas & higher-order functions

A **lambda** is a small anonymous function written inline: `lambda args: expression`. A **higher-order function** takes a function as an argument (or returns one) — `map`, `filter`, `reduce`, and `sorted` are the common ones.

In [ ]:
from functools import reduce

counts = [120, 95, 130, 88, 0, 145]
sites  = ["Riverside", "Hilltop", "Downtown"]

doubled = list(map(lambda c: c * 2, counts))     # transform each
nonzero = list(filter(lambda c: c > 0, counts))  # keep where True
product = reduce(lambda a, b: a * b, [1, 2, 3, 4])  # cumulative -> 24
by_len  = sorted(sites, key=lambda s: len(s))    # sort by a derived key

print(doubled)
print(nonzero)
print(product)
print(by_len)

In practice, a comprehension is usually clearer than `map`/`filter`. The place a lambda really earns its keep is as the `key=` to `sorted`, `min`, or `max`.

### Exercise 2 — Keyword arguments *(5 min)*

Create a function that takes a required `name` plus two keyword parameters with sensible defaults (for example `role` and `site`). It should **return** a sentence (do not just `print` inside the function). Then call it and print the result.

In [ ]:
# Your work here


<details>
<summary>Solution</summary>

```python
def describe(name, role="staff", site="unknown"):
    return f"{name} works as a {role} at the {site} clinic."

sentence = describe("Maria", role="nurse", site="Riverside")
print(sentence)
```

**Why this works.** The defaults make `role` and `site` optional, so the function is usable with just a name. Returning the string instead of printing it hands control back to the caller — they can print it, store it, log it, or build it into something larger. A function that only prints can't be reused that way.

</details>

### Exercise 3 — Lambda as a sort key *(10 min)*

Given a list of `(site, doses)` tuples, use `sorted` with a `lambda` to rank the sites from most doses to fewest.

`data = [("Riverside", 947), ("Hilltop", 612), ("Downtown", 388), ("Eastgate", 720)]`

In [ ]:
data = [("Riverside", 947), ("Hilltop", 612), ("Downtown", 388), ("Eastgate", 720)]

# Your work here


<details>
<summary>Solution</summary>

```python
ranked = sorted(data, key=lambda pair: pair[1], reverse=True)
print(ranked)
```

**Why this works.** `sorted` needs to know *what* to sort on; `key=` takes a function that, given one element, returns the value to compare. Here the lambda receives each `(site, doses)` tuple and returns `pair[1]` — the dose count. `reverse=True` flips the order to high-to-low. This is the single most common real-world use of a lambda.

</details>

## 8. A practical example with pandas

A preview of where this is heading: use functions to tidy a patient table.

> **Data note.** In class this becomes a one-liner — `df = pd.read_csv(DATA_URL)` — pointing at the course's public data repo. Until that repo is live, the cell below builds a small inline sample of the same shape so the notebook runs as-is. Replace it with the `read_csv` line once the repo URL is set.

In [ ]:
import pandas as pd

In [ ]:

# Once the shared data repo is live, this whole cell becomes:
#   DATA_URL = "https://raw.githubusercontent.com/<org>/<repo>/main/patients.csv"
#   df = pd.read_csv(DATA_URL)
df = pd.DataFrame({
    "ID":     [4172, 4188, 4205, 4219],
    "Age":    [58, 47, 63, 39],
    "A1C":    [7.4, 6.1, 9.2, 5.5],
    "BMI":    [31.2, 27.8, 34.1, 24.6],
    "Smoker": [True, False, True, False],
    "Site":   ["Riverside", "Hilltop", "Riverside", "Downtown"],
    "Sex":    ["F", "M", "F", "M"],
})
df.head()

A function to lowercase the column names. Note it operates on its **parameter** `names`, not on a global — so it works on any set of names you hand it.

In [ ]:
def lower_columns(names):
    """Return the given column names, lowercased."""
    return [name.lower() for name in names]

df.columns = lower_columns(df.columns)
df.head()

In real pandas you'd reach for the vectorized one-liner — but writing it as a function first shows what that shortcut is doing.

In [ ]:
# Pythonic / pandas-native equivalent:
df.columns = df.columns.str.lower()
df.columns.tolist()

A second function that groups column names by broad dtype and returns a **dict** — clearer than returning five separate values. Again, it uses its `data` parameter throughout.

In [ ]:
def columns_by_dtype(data):
    """Group a DataFrame's column names by broad dtype."""
    return {
        "numeric": data.select_dtypes("number").columns.tolist(),
        "integer": data.select_dtypes("int").columns.tolist(),
        "float":   data.select_dtypes("float").columns.tolist(),
        "boolean": data.select_dtypes("bool").columns.tolist(),
        "text":    data.select_dtypes(include=["object", "string"]).columns.tolist(),
    }

columns_by_dtype(df)

> Note: in pandas 3.x, text columns are the new `str`/`string` dtype (they were `object` in 2.x). Selecting `include=["object", "string"]` catches text columns under either version.

## Wrap-up

You can now define functions with flexible arguments, use `return` deliberately, reason about scope, avoid the mutable-default trap, and apply lambdas where they read best.

**Next:** NumPy — fast numerical arrays, the foundation under pandas.